# Chinook SQL Agent（LangChain + LangGraph 版）

本 notebook 演示用 **LangChain + LangGraph** 搭建的完整 SQL Agent，与原手写编排版（`06_SQL-Agent-Project`）对外契约完全一致：

| 层 | 原版 | 本项目 |
|---|---|---|
| LLM 调用 | OpenAI SDK 直调 | `langchain-openai` 的 `ChatOpenAI`（OpenAI 兼容接口接入 DeepSeek） |
| 提示词 | `string.Template($var)` | `ChatPromptTemplate`（`{var}`，模板文本逐字一致） |
| 选表输出 | 手写 JSON 解析 | `with_structured_output(Pydantic schema)`，失败降级 JSON |
| 流程编排 | `agent.run()` 手写 if/for | `langgraph` `StateGraph` 13 节点状态机 + 条件边 |

工作流：

```text
中文问题
  -> 意图门禁 + 语义预检（LangGraph 条件边拒答短路）
  -> 选表（LLM 结构化输出 + 外键图补全）
  -> SQL 生成（ChatPromptTemplate: schema + 样例 + 值域 + few-shot）
  -> 校验（sqlglot 静态 + EXPLAIN 动态 + 只读白名单）
  -> 失败则改写（错误反思循环，LangGraph 改写环，最多 N 次）
  -> 只读执行（超时 + 限行）
  -> 空结果归因 + 受约束中文汇总
```

运行前请在项目根目录 `.env` 中配置 `DEEPSEEK_API_KEY`。

In [1]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "sql_agent").exists():
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import pandas as pd
from IPython.display import Markdown, display

from sql_agent.catalog import DBCatalog
from sql_agent.config import settings
from sql_agent.llm import LLM

print("数据库:", settings.db_path)
print("模型:", settings.model)
print("API Key 已配置:", bool(settings.api_key))
assert settings.api_key, "请在 .env 中配置 DEEPSEEK_API_KEY"

llm = LLM(settings.api_key, settings.base_url, settings.model)

数据库: D:\BaiduNetdiskDownload\ai\直播课程\第十二周\第12周：多模态大模型和RAG应用\Week12_2026Q2\sql-agent-langchain\data\chinook.db
模型: deepseek-chat
API Key 已配置: True


## ⓪ LangGraph 状态机可视化

`build_graph(deps)` 每次请求编译一个 13 节点的 `StateGraph`。用 `draw_mermaid()` 导出拓扑：
条件边 `intent_route / semantic_route / validate_route / execute_route` 负责拒答短路、改写环与预算熔断。

In [2]:
from sql_agent.agent import SQLAgent
from sql_agent.budget import Budget
from sql_agent.graph import GraphDeps, build_graph

catalog = DBCatalog(settings.db_path, settings.cache_dir).load()
agent = SQLAgent(catalog=catalog, llm=llm)

deps = GraphDeps(
    settings=agent.settings,
    catalog=agent.catalog,
    semantic=agent.semantic,
    llm=agent.llm,
    intent_gate=agent.intent_gate,
    selector=agent.selector,
    generator=agent.generator,
    validator=agent.validator,
    rewriter=agent.rewriter,
    executor=agent.executor,
    summarizer=agent.summarizer,
    budget=Budget(),
    data_until=agent.data_until,
)
graph = build_graph(deps)
mermaid = graph.get_graph().draw_mermaid()
display(Markdown("```mermaid\n" + mermaid + "\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	input_check(input_check)
	intent_gate(intent_gate)
	semantic_bind(semantic_bind)
	select_tables(select_tables)
	semantic_resolve(semantic_resolve)
	generate_sql(generate_sql)
	validate_sql(validate_sql)
	rewrite_sql(rewrite_sql)
	execute_sql(execute_sql)
	classify_empty(classify_empty)
	sanity_fill(sanity_fill)
	summarize(summarize)
	finalize(finalize)
	__end__(<p>__end__</p>)
	__start__ --> input_check;
	input_check --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## ① 数据库解析

采集表、字段、类型、主键、外键、行数、样例数据、字段取值分布，并生成提示词用的 DDL。

In [3]:
print(f"共解析到 {len(catalog.table_names)} 张业务表")
for name in catalog.table_names:
    print(f"- {name}: {catalog.tables[name].row_count} 行")

fks = [
    {
        "主表": fk.table,
        "外键列": ",".join(fk.columns),
        "引用表": fk.referred_table,
        "引用列": ",".join(fk.referred_columns),
    }
    for fk in catalog.foreign_keys
]
display(pd.DataFrame(fks))
display(Markdown("```sql\n" + catalog.schema_text(["tracks"]) + "\n```"))

共解析到 11 张业务表
- albums: 347 行
- artists: 275 行
- customers: 59 行
- employees: 8 行
- genres: 25 行
- invoice_items: 2240 行
- invoices: 412 行
- media_types: 5 行
- playlist_track: 8715 行
- playlists: 18 行
- tracks: 3503 行


,主表,外键列,引用表,引用列
0,albums,ArtistId,artists,ArtistId
1,customers,SupportRepId,employees,EmployeeId
2,employees,ReportsTo,employees,EmployeeId
3,invoice_items,TrackId,tracks,TrackId
4,invoice_items,InvoiceId,invoices,InvoiceId
5,invoices,CustomerId,customers,CustomerId
6,playlist_track,TrackId,tracks,TrackId
7,playlist_track,PlaylistId,playlists,PlaylistId
8,tracks,MediaTypeId,media_types,MediaTypeId
9,tracks,GenreId,genres,GenreId


```sql
-- 数据范围: 2009-01-01 ~ 2013-12-22

-- tracks (3503 行)
CREATE TABLE "tracks" (
  "TrackId" INTEGER PRIMARY KEY,,
  "Name" NVARCHAR(200),,
  "AlbumId" INTEGER,  -- FK -> albums.AlbumId,
  "MediaTypeId" INTEGER,  -- FK -> media_types.MediaTypeId,
  "GenreId" INTEGER,  -- FK -> genres.GenreId,
  "Composer" NVARCHAR(220),,
  "Milliseconds" INTEGER,,
  "Bytes" INTEGER,,
  "UnitPrice" NUMERIC(10,2)
);

样例行:
| TrackId | Name | AlbumId | MediaTypeId | GenreId | Composer | Milliseconds | Bytes | UnitPrice |
|---|---|---|---|---|---|---|---|---|
| 1 | For Those About To Rock (We Salute You) | 1 | 1 | 1 | Angus Young, Malcolm Young, Brian Johnson | 343719 | 11170334 | 0.99 |
| 2 | Balls to the Wall | 2 | 2 | 1 | NULL | 342562 | 5510424 | 0.99 |
| 3 | Fast As a Shark | 3 | 2 | 1 | F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman | 230619 | 3990994 | 0.99 |

值域提示:
- TrackId: 范围 [1, 3503]
- AlbumId: 范围 [1, 347]
- MediaTypeId: 范围 [1, 5]
- GenreId: 范围 [1, 25]
- Milliseconds: 范围 [1071, 5286953]
- Bytes: 范围 [38747, 1059546140]
- UnitPrice: 范围 [0.99, 1.99]
```

## ② Table 选择（with_structured_output）

关键词提示 + LLM 选表，再用外键图补全多对多中间表。

LangChain 改造点：LLM 输出经 `with_structured_output(SelectionOut)` 直接解析为 Pydantic 对象（`tables / reason / confidence`），解析失败时自动降级为 JSON 文本解析。

In [4]:
from sql_agent.selector import SelectionOut, TableSelector

print("结构化输出 schema:", SelectionOut.model_json_schema())
print()

question = "哪个歌单包含最多的摇滚歌曲？"
selector = TableSelector(catalog, llm)
selection = selector.select(question)

print("问题:", question)
print("选中表:", selection.tables)
print("理由:", selection.reason)
print("是否兜底:", selection.used_fallback)

结构化输出 schema: {'description': '选表结构化输出 schema（LangChain with_structured_output 用）。', 'properties': {'tables': {'items': {'type': 'string'}, 'title': 'Tables', 'type': 'array'}, 'reason': {'default': '', 'title': 'Reason', 'type': 'string'}, 'confidence': {'default': 'high', 'title': 'Confidence', 'type': 'string'}}, 'title': 'SelectionOut', 'type': 'object'}



问题: 哪个歌单包含最多的摇滚歌曲？
选中表: ['genres', 'playlist_track', 'playlists', 'tracks']
理由: 需要连接歌单、歌单曲目、曲目和流派表，按摇滚流派统计各歌单的歌曲数量。
是否兜底: False


## ③ SQL 生成（ChatPromptTemplate）

使用选中表的 DDL、样例行、值域提示和 few-shot 示例生成 SQLite SQL。

LangChain 改造点：提示词由 `sql_agent.prompts.GENERATOR_PROMPT`（`ChatPromptTemplate`）渲染，`format_messages()` 产出 `HumanMessage` 列表直接送入 `ChatOpenAI`。

In [5]:
from sql_agent.generator import SQLGenerator

schema = catalog.schema_text(selection.tables)
generator = SQLGenerator(llm)
sql = generator.generate(question, schema)
print(sql)

SELECT p.Name AS playlist, COUNT(*) AS rock_track_count
FROM playlists p
JOIN playlist_track pt ON p.PlaylistId = pt.PlaylistId
JOIN tracks t ON pt.TrackId = t.TrackId
JOIN genres g ON t.GenreId = g.GenreId
WHERE g.Name = 'Rock'
GROUP BY p.PlaylistId, p.Name
ORDER BY rock_track_count DESC
LIMIT 1;


## ④ SQL 校验

静态：sqlglot 解析，检查单语句、只读类型、表/列是否存在；动态：EXPLAIN 编译校验。

In [6]:
from sql_agent.validator import SQLValidator

validator = SQLValidator(catalog)

result = validator.validate(sql)
print("通过:", result.ok)
print("阶段:", result.stage)
print("信息:", result.error)

# 反例：不存在的列
bad_validation = validator.validate("SELECT NotAColumn FROM tracks")
print("\n反例通过:", bad_validation.ok)
print("反例阶段:", bad_validation.stage)
print("反例信息:", bad_validation.error)

通过: True
阶段: ok
信息: 静态/动态校验通过

反例通过: False
反例阶段: unknown_column
反例信息: 列不存在: NotAColumn


## ⑤ SQL 改写

把「问题 + 原 SQL + 错误信息 + 历史尝试」交给大模型修复，修复后重新校验。
LangGraph 中这一步在校验/执行节点之间形成改写环（最多 `SQL_AGENT_MAX_REWRITE` 次）。

In [7]:
from sql_agent.rewriter import SQLRewriter

rewriter = SQLRewriter(llm)

rewrite_question = "一共有多少首歌曲？"
rewrite_schema = catalog.schema_text(["tracks"])
bad_sql = "SELECT Foo FROM tracks"

bad_val = validator.validate(bad_sql)
print("原始 SQL:", bad_sql)
print("错误:", bad_val.stage, bad_val.error)

fixed_sql = rewriter.rewrite(
    rewrite_question,
    rewrite_schema,
    bad_sql,
    f"{bad_val.stage}: {bad_val.error}",
    history=[],
)
print("\n改写后 SQL:")
print(fixed_sql)
print("\n再次校验:", validator.validate(fixed_sql))

原始 SQL: SELECT Foo FROM tracks
错误: unknown_column 列不存在: Foo



改写后 SQL:
SELECT COUNT(TrackId) AS TrackCount FROM tracks

再次校验: ValidationResult(ok=True, stage='ok', error='静态/动态校验通过', code='')


## ⑥ SQL 执行

只读连接 + 超时中断 + 最多返回 N 行，避免写操作和超大结果。

In [8]:
from sql_agent.executor import SQLExecutor

executor = SQLExecutor(settings.db_path, max_rows=50, timeout_s=settings.query_timeout_s)
exec_result = executor.execute(sql)

print("执行成功:", exec_result.ok)
print("返回行数:", exec_result.row_count)
print("是否截断:", exec_result.truncated)
print("耗时(ms):", round(exec_result.elapsed_ms, 2))

if exec_result.ok:
    display(pd.DataFrame(exec_result.rows, columns=exec_result.columns))
else:
    print("错误:", exec_result.error)

执行成功: True
返回行数: 1
是否截断: False
耗时(ms): 5.49


,playlist,rock_track_count
0,Music,1297


## ⑦ 结果汇总

把问题、SQL 和结果表交给大模型，生成中文结论；禁止编造数字（结构化 claims + 程序化 grounding）。

In [9]:
from sql_agent.summarizer import ResultSummarizer

summarizer = ResultSummarizer(llm)
answer = summarizer.summarize(question, sql, exec_result)
print(answer)

SummaryResult(answer='包含最多摇滚歌曲的歌单是 Music，共有 1297 首摇滚歌曲。', claims=[Claim(text='歌单 Music 包含 1297 首摇滚歌曲', value=1297.0, source='row[0].rock_track_count', tolerance=0.01)], grounding_ok=True, used_fallback=False, error_code='', warnings=[])


## 完整 Agent 端到端（LangGraph.invoke）

一次 `SQLAgent.run()` 内部完成：`Budget` 构建 → `GraphDeps` 组装 → `build_graph()` 编译 → `graph.invoke(state)`（`recursion_limit = 2*max_rewrites+20` 兜底）→ state 转 `AgentResult` → JSONL 审计。

`AgentResult` 的字段与错误码体系和原手写编排版完全一致。

In [10]:
agent_result = agent.run("消费总额最高的前5位客户是谁？")

for step in agent_result.steps:
    print(f"[{step.status}] {step.name} ({step.elapsed_ms:.0f} ms)")
    print(step.detail)
    print("-" * 60)

print("最终 SQL:")
print(agent_result.sql)
print("\n答案:")
print(agent_result.answer)
print("\ntrace_id:", agent_result.trace_id)
print("预算快照:", agent_result.budget)
display(agent_result.to_dataframe())

[ok] ① 意图门禁 (0 ms)
intent=analytics
------------------------------------------------------------
[ok] ② 语义绑定 (0 ms)
指标/时间预检通过
------------------------------------------------------------
[ok] ③ 表选择 (989 ms)
选中表: customers, invoices
理由: 需要从customers获取客户信息，并按invoices中的Total汇总消费总额排序取前5
------------------------------------------------------------
[ok] ④ 语义解析 (1 ms)
metric=revenue; time=全量数据（2009-01-01 ~ 2013-12-22）
------------------------------------------------------------
[ok] ⑤ SQL 生成 (708 ms)
SELECT CustomerId, ROUND(SUM(Total), 2) AS revenue
FROM invoices
GROUP BY CustomerId
ORDER BY revenue DESC
LIMIT 5;
------------------------------------------------------------
[ok] ⑥ SQL 校验 (4 ms)
静态/动态校验通过
------------------------------------------------------------
[ok] ⑦ SQL 执行 (3 ms)
返回 5 行，耗时 3.4 ms
------------------------------------------------------------
[ok] ⑨ 结果汇总 (1654 ms)
按销售额（SUM(invoices.Total)，USD，账单粒度，全量数据 2009-01-01 ~ 2013-12-22）统计，消费总额最高的前5位客户为：客户6（49.62美元）、客户26（47.62美元）、客户57

,CustomerId,revenue
0,6,49.62
1,26,47.62
2,57,46.62
3,45,45.62
4,46,45.62


## 拒答与错误码演示

意图门禁 / 语义层拒答不消耗 LLM 调用：非查询意图（E1005）、未定义指标（E7005）直接短路到 finalize。

In [11]:
for q in ["帮我删掉 invoices 表", "上周客单价是多少？"]:
    r = agent.run(q)
    print(f"Q: {q}")
    print(f"   status={r.status} error_code={r.error_code} budget_calls={r.budget['calls']}")
    print(f"   {r.user_message}")

Q: 帮我删掉 invoices 表
   status=error error_code=E1004 budget_calls=0
   我只能做只读查询，不能修改数据；当前数据只覆盖 Chinook 音乐商店的客户、发票、曲目等。
Q: 上周客单价是多少？
   status=error error_code=E7005 budget_calls=0
   当前未定义指标「客单价」，因此不能生成该口径的 SQL。可以改用：销售额、人均消费。


## 批量评测

`eval/gold.jsonl` 共 130 题（dev/holdout/boundary/entity/intent/security 六分片）。下面只跑前 5 条快速验证。

In [12]:
# 完整评测：python eval/evaluate.py
import subprocess

completed = subprocess.run(
    [sys.executable, "eval/evaluate.py", "--limit", "5"],
    capture_output=True,
    text=True,
    encoding="utf-8",
)
print(completed.stdout[-3000:])
if completed.stderr:
    print(completed.stderr[-1000:])

gold 校验通过: 5 条
[001/005] PASS d01  ok            7177ms | 数据库里一共有多少首歌曲？
[002/005] PASS d02  ok           14204ms | 一共有多少位艺人？
[003/005] PASS d03  ok            2719ms | 一共有多少张专辑？
[004/005] PASS d04  ok            5313ms | 每个流派有多少首歌曲？按歌曲数量从高到低排序。
[005/005] PASS d05  ok            4307ms | 歌曲数量最多的前5个流派是哪些？

总体准确率 : 5/5 (100.0%) CI95=[56.5%, 100.0%]
各次运行   : [5, 5, 5] 中位数=5
  dev      : 5/5 (100.0%) runs=[5, 5, 5]
选表       : precision=100.0% recall=100.0% fallback=0 (precision=None)
grounding  : 15/15 (100.0%) 模板回退=0
gate:
  [FAIL] dev_total: 5/5 (门槛 28)
  [PASS] selection_precision: 100.0% (门槛 85%)
  [PASS] selection_recall: 100.0% (门槛 95%)
  [PASS] grounding_100: 100.0%
  [PASS] regression_correct_drop: 较基线少正确 0 题 (阻断阈值 2)
  [PASS] regression_core_tags: 核心标签回退 0 个: -

评测结论: BLOCK
详细结果已保存: D:\BaiduNetdiskDownload\ai\直播课程\第十二周\第12周：多模态大模型和RAG应用\Week12_2026Q2\sql-agent-langchain\eval\results.json



## 启动可视化 UI

```bash
streamlit run app.py
```

UI 支持：聊天式提问、步骤轨迹、SQL 查看、结果表格、自动柱状图、示例问题。